In [1]:
import kagglehub
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
import os

# Download latest version
# path = kagglehub.dataset_download("harunrai/digital-wallet-transactions")

# print("Path to dataset files:", path)

## Data processing with manipulation

In [2]:
# Import data from downloaded CSV file

df_digital = pd.read_csv("/Users/wildanhidayat/.cache/kagglehub/datasets/harunrai/digital-wallet-transactions/versions/1/digital_wallet_transactions.csv")

In [3]:
# Standardize column names
df_digital.columns = df_digital.columns.str.lower().str.replace(' ', '_')

In [5]:
df_digital

,idx,transaction_id,user_id,transaction_date,product_category,product_name,merchant_name,product_amount,transaction_fee,cashback,loyalty_points,payment_method,transaction_status,merchant_id,device_type,location
0,1,4dac3ea3-6492-46ec-80b8-dc45c3ad0b14,USER_05159,2023-08-19 03:32,Rent Payment,2BHK Flat Deposit,Airbnb,1525.39,36.69,19.19,186,Debit Card,Successful,MERCH_0083,iOS,Urban
1,2,a903ed9f-eb84-47e7-b8aa-fd1786c919cf,USER_07204,2023-08-19 04:37,Gas Bill,Commercial Gas Connection,Adani Gas,1495.40,28.19,89.99,182,UPI,Successful,MERCH_0163,iOS,Urban
2,3,2a393013-733c-4add-9f09-bed1eeb33676,USER_00903,2023-08-19 05:52,Bus Ticket,Semi-Sleeper,MakeMyTrip Bus,1267.71,11.36,95.70,994,UPI,Successful,MERCH_0320,iOS,Urban
3,4,9a07ad19-4673-4794-9cd2-9b139f39c715,USER_01769,2023-08-19 06:35,Internet Bill,4G Unlimited Plan,Airtel Broadband,9202.63,6.41,82.24,409,Debit Card,Successful,MERCH_0194,Android,Urban
4,5,76418260-c985-4011-979d-0914604d0d68,USER_03544,2023-08-19 06:36,Loan Repayment,Home Loan EMI,Axis Bank,3100.58,41.15,40.47,837,Debit Card,Successful,MERCH_0504,Android,Urban
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,4996,2cad1456-e6dd-4d0f-9463-a071a462c2ae,USER_03918,2024-08-17 19:39,Internet Bill,4G Unlimited Plan,BSNL Broadband,9496.45,37.25,87.47,284,UPI,Successful,MERCH_0203,Android,Rural
4996,4997,df3984f6-cf9d-418f-ad2e-f9d4fff69cd5,USER_09281,2024-08-17 19:42,Education Fee,Python Programming Course,Unacademy,710.40,14.01,55.24,538,Credit Card,Successful,MERCH_0480,iOS,Urban
4997,4998,1c4cd450-785d-4f4c-a08d-cb7c17db9b0c,USER_05939,2024-08-17 23:24,Movie Ticket,Avengers: Endgame,PVR Cinemas,2750.06,48.69,72.10,652,Bank Transfer,Successful,MERCH_0411,Android,Suburban
4998,4999,f88d6fb3-d747-4963-82f3-92e181a93771,USER_09925,2024-08-18 00:28,Mobile Recharge,Data Pack 50GB,BSNL,9976.19,21.96,40.44,614,UPI,Successful,MERCH_0524,iOS,Suburban


In [ ]:
# Simulate realistic skewed API latency and integration type by adding api_latency_ms
np.random.seed(42)

# Gamma distribution: shape(k)=2.0, scale(theta)=75 -> long right tail
# Base overhead = 80ms; base P50 ~ 180-230ms, tail can go up to 800ms or more
base_latency = np.random.gamma(shape=2.0, scale=75, size=(len(df_digital))) + 80

# Occasional 5% network or rail timeouts (adding 400-800ms)
timeout_spikes = np.where(
    np.random.rand(len(df_digital)) < 0.05, np.random.uniform(400, 800, size=(len(df_digital))), 0
)
df_digital["api_latency_ms"] = np.round(base_latency + timeout_spikes).astype(int)


In [7]:
# Simulate integration types to evaluate technical performance across different integration methods
integration_options = ["shopify_plugin", "raw_api", "hosted_page"]
integration_probs = [0.5, 0.35, 0.15]
df_digital["integration_type"] = np.random.choice(
    integration_options, size=len(df_digital), p=integration_probs
)

In [9]:
df_digital

,idx,transaction_id,user_id,transaction_date,product_category,product_name,merchant_name,product_amount,transaction_fee,cashback,loyalty_points,payment_method,transaction_status,merchant_id,device_type,location,api_latency_ms,integration_type
0,1,4dac3ea3-6492-46ec-80b8-dc45c3ad0b14,USER_05159,2023-08-19 03:32,Rent Payment,2BHK Flat Deposit,Airbnb,1525.39,36.69,19.19,186,Debit Card,Successful,MERCH_0083,iOS,Urban,260,raw_api
1,2,a903ed9f-eb84-47e7-b8aa-fd1786c919cf,USER_07204,2023-08-19 04:37,Gas Bill,Commercial Gas Connection,Adani Gas,1495.40,28.19,89.99,182,UPI,Successful,MERCH_0163,iOS,Urban,192,raw_api
2,3,2a393013-733c-4add-9f09-bed1eeb33676,USER_00903,2023-08-19 05:52,Bus Ticket,Semi-Sleeper,MakeMyTrip Bus,1267.71,11.36,95.70,994,UPI,Successful,MERCH_0320,iOS,Urban,184,raw_api
3,4,9a07ad19-4673-4794-9cd2-9b139f39c715,USER_01769,2023-08-19 06:35,Internet Bill,4G Unlimited Plan,Airtel Broadband,9202.63,6.41,82.24,409,Debit Card,Successful,MERCH_0194,Android,Urban,184,hosted_page
4,5,76418260-c985-4011-979d-0914604d0d68,USER_03544,2023-08-19 06:36,Loan Repayment,Home Loan EMI,Axis Bank,3100.58,41.15,40.47,837,Debit Card,Successful,MERCH_0504,Android,Urban,429,raw_api
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,4996,2cad1456-e6dd-4d0f-9463-a071a462c2ae,USER_03918,2024-08-17 19:39,Internet Bill,4G Unlimited Plan,BSNL Broadband,9496.45,37.25,87.47,284,UPI,Successful,MERCH_0203,Android,Rural,161,shopify_plugin
4996,4997,df3984f6-cf9d-418f-ad2e-f9d4fff69cd5,USER_09281,2024-08-17 19:42,Education Fee,Python Programming Course,Unacademy,710.40,14.01,55.24,538,Credit Card,Successful,MERCH_0480,iOS,Urban,528,hosted_page
4997,4998,1c4cd450-785d-4f4c-a08d-cb7c17db9b0c,USER_05939,2024-08-17 23:24,Movie Ticket,Avengers: Endgame,PVR Cinemas,2750.06,48.69,72.10,652,Bank Transfer,Successful,MERCH_0411,Android,Suburban,422,shopify_plugin
4998,4999,f88d6fb3-d747-4963-82f3-92e181a93771,USER_09925,2024-08-18 00:28,Mobile Recharge,Data Pack 50GB,BSNL,9976.19,21.96,40.44,614,UPI,Successful,MERCH_0524,iOS,Suburban,128,shopify_plugin


In [16]:
# Calculate financial metrics
df_digital["transaction_date"] = pd.to_datetime(df_digital["transaction_date"])
df_digital["transaction_day"] = df_digital["transaction_date"].dt.date

# Calculate take rate % per transaction
df_digital["gpv_amount"] = df_digital["product_amount"].astype(float)
df_digital["gross_revenue"] = df_digital["transaction_fee"].astype(float)

df_digital["take_rate"] = np.where(
    df_digital["gpv_amount"] > 0,
    np.round(df_digital["gross_revenue"] / df_digital["gpv_amount"], 4),
    0.0
)

# Successful transactions alias
df_digital["is_success"] = df_digital["transaction_status"].apply(
    lambda x: 1 if str(x).lower() == "successful" else 0
)

### Create the data lake

In [ ]:
# Database connection
DB_USER = "postgres"
DB_PASSWORD = ""
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "vehicle_insights"

In [ ]:
# Engine connection to the database
engine = create_engine(f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}")